<div style="background: #86d1f1ff; border-radius: 5px; padding: 1rem; margin-bottom: 1rem">
<img src="https://store.utec.edu.pe/files/Recursos/logo-utec-h.png" alt="Banner" width="150" />   
<div style="font-weight: bold; color: #434549ff; float: right "><u style="font-size: 28px;">Base de Datos II</u> <br />
<span style="float:right"> Profesor Heider Sanchez</span> <br /> 
<span style="float:right">  2026 - 1 </span>   
</div> </div>

# Laboratorio 8.1: Procesamiento de Textos y Bag of Words

> **Prof. Heider Sanchez**  

##  Introducción
Este laboratorio tiene como objetivo el análisis y búsqueda de documentos textuales utilizando procesamiento de lenguaje natural (NLP) y una base de datos PostgreSQL. Se trabajará paso a paso desde la extracción de los textos hasta la aplicación búsquedas booleanas.


### Objetivos
- Configurar la tabla en PostgreSQL y carga de datos.
- Desde Python leer los textos desde PostgreSQL.
- Realizar el procesamiento de textos: convertir a minúscula, tokenización, stopwords, stemming y frecuencia de términos.
- Almacenar los Bag of Words en la base de datos en formato JSON.
- Realizar búsquedas de documentos similares a una consulta booleana (conectores AND, OR y AND-NOT).


### Requisitos previos

- Tener instalado PostgreSQL en su computadora (ultima versión)
- Tener instalado las siguientes dependencias en Python:

    ```bash
    pip install psycopg2-binary nltk scikit-learn pandas
    ```

- Opcionalmente descargar los recursos de NLTK:

    ```python
    import nltk
    nltk.download('punkt')
    ```


## 1. (2 puntos) Configurar la tabla en PostgreSQL y carga de datos


### Crear las tablas

Crear la tabla en PostgreSQL para almacenar los textos de noticias y el bag of words:

```sql
CREATE TABLE noticias (
    id SERIAL PRIMARY KEY,
    url TEXT,
    contenido TEXT,
    categoria VARCHAR(50),
    bag_of_words JSONB
);
```

Además, crear una tabla para almacenar los stopwords

```sql
CREATE TABLE stopwords (
    id SERIAL PRIMARY KEY,
    word TEXT UNIQUE NOT NULL
);
```

### Carga de datos en PostgreSQL

Proceder a cargar el dataset de noticias `news_es.csv` y el dataset de stopwords `stoplist_es.txt`.

### Leer desde PostgreSQL con Python

Completar la función para conectarte a PostgreSQL y leer los datos:

In [14]:
%pip install pandas psycopg2-binary nltk scikit-learn

  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
Using cached nltk-3.9.4-py3-none-any.whl (1.6 MB)
Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl (8.0 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 32] El proceso no tiene acceso al archivo porque está siendo utilizado por otro proceso: 'c:\\Users\\josue\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\nltk\\parse\\transitionparser.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import psycopg2
import pandas as pd

def connect_db():
    conn = psycopg2.connect(
        dbname="Information Retrieval",
        user="postgres",
        password="123456",
        host="localhost",
        port="5432"
    )
    return conn

def fetch_data():
    conn = connect_db()
    query = "SELECT id, contenido FROM noticias;"
    df = pd.read_sql(query, conn)
    conn.close()
    return df

noticias_df = fetch_data()
print(f"¡Carga completada! Se descargaron {len(noticias_df)} filas.")

¡Carga completada! Se descargaron 1217 filas.


C:\Users\josue\AppData\Local\Temp\ipykernel_18768\1544173908.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached nltk-3.9.4-py3-none-any.whl (1.6 MB)
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.8/2.8 MB 5.6 MB/s eta 0:00:01
   ----------- ---------------------------- 0.8/2.8 MB 5.6 MB/s eta 0:00:01
   --------------- ------------------------ 1.0/2.8 MB 1.9 MB/s eta 0:00:01
   --------------- ------------------------ 1.0/2.8 MB 1.9 MB/s eta 0:00:01
   ------------------- -------------------- 1.3/2.8 MB 1.3 MB/s eta 0:00:02
   ------------------- -------------------- 1.3/2.8 MB 1.3 MB/s eta 0:00:02
   ---------------------- ----------------- 1.6/2.8 MB 1.1 MB/s eta 0:00:02
   ---------------------- -

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. (4 puntos) Preprocesamiento de texto

Implementar la función `preprocess` que reciba un texto y realice los siguiente:
- Convertir el texto a minuscula.
- Tokenización.
- Eliminación de stopwords
- Stemming (raíz de las palabras)

In [22]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\josue\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\josue\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [25]:
def fetch_stopwords():
    conn = connect_db()
    query = "SELECT word FROM stopwords;"
    df_words = pd.read_sql(query, conn)
    conn.close()
    return set(df_words['word'].tolist())

# Creamos la variable global que le falta a tu función
stopwords_es = fetch_stopwords()
print(f"¡Listo! Se cargaron {len(stopwords_es)} stopwords desde la base de datos.")

¡Listo! Se cargaron 608 stopwords desde la base de datos.


C:\Users\josue\AppData\Local\Temp\ipykernel_18768\280612892.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_words = pd.read_sql(query, conn)


In [26]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

# Inicializamos el stemmer en español para reducir las palabras a su raíz léxica
stemmer = SnowballStemmer('spanish')

def preprocess(text):
    """
    Recibe un texto crudo y realiza: minúsculas, tokenización, 
    eliminación de stopwords y stemming.
    """
    if not text:
        return []
    
    # 1. Convertir el texto a minúscula
    text = text.lower() 
    
    # 2. Tokenización (separar en palabras y signos)
    tokens = word_tokenize(text) 
    
    # 3. Eliminación de stopwords, números y símbolos
    # 'token.isalpha()' asegura que solo se queden palabras (elimina 123, #, @, ¡, ?, etc.)
    cleaned_tokens = [
        token for token in tokens 
        if token.isalpha() and token not in stopwords_es
    ] 
    
    # 4. Stemming (reducir las palabras a su raíz)
    stemmed_tokens = [stemmer.stem(token) for token in cleaned_tokens] 
    
    return stemmed_tokens




In [27]:
# Ejecutar la prueba de preprocesamiento de la guía
print("--- Prueba de Preprocesamiento ---")
print(preprocess("¡Hola! Esto es una prueba, con números 123 y símbolos #@$%$. ¿Funcionará bien?")) 

--- Prueba de Preprocesamiento ---
['prueb', 'numer', 'simbol']


Luego, implementar una función para calcular la frecuencia de términos:

In [28]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

def compute_bow(text):
    """
    Recibe un texto, lo preprocesa y calcula la frecuencia de 
    cada término (Bag of Words) retornando un diccionario.
    """
    # Obtenemos la lista de raíces (tokens) ya procesada
    tokens = preprocess(text)
    
    # Calculamos la frecuencia de términos
    bow = {}
    for token in tokens:
        bow[token] = bow.get(token, 0) + 1
        
    return bow




In [29]:
compute_bow("Esta es una prueba con prueba y más prueba. ¡Prueba!. Programadores programadores programando.")

{'prueb': 3, 'mas': 1, 'program': 3}

## 3. (3 puntos) Actualizar la base de datos con los Bag of Words

Guardar el resultado del Bag of Words en la columna `bag_of_words` de la tabla:

In [ ]:
def update_bow_in_db(dataframe):
    # Implementar la función de actualización en la base de datos aquí
    pass

update_bow_in_db(noticias_df)

In [ ]:
def fetch_data_bow():
    # Mostrar las primeras 10 filas de la tabla noticias_bow
    pass


## 4. (8 puntos) Consulta booleana con filtrado por keywords

Antes de aplicar el filtrado desde Python, es importante entender cómo funciona la consulta de una clave dentro de una columna JSONB en PostgreSQL. 

### Ejemplo consulta SQL en JSON:

```sql
SELECT * FROM noticias WHERE bag_of_words ? 'keyword';
```

Esta consulta selecciona todos los registros en los que el `bag_of_words` (formato JSONB) contiene una clave igual a `'keyword'`. El operador `?` verifica la existencia de una clave dentro de un JSON.

### Consulta booleana 
Implementar una función que permita parsear una consulta textual con conectores AND, OR y AND-NOT y con ello se aplique el filtro correspondiente directamente desde la base de datos. 


In [ ]:
def apply_boolean_query(query):
    # Construir la condición de búsqueda a partir de la query booleana
    # Ejecutar la consulta en la base de datos
    # Retornar un DataFrame con los resultados
    return pd.DataFrame()  

### Pruebas funcionales

Realizar al menos 8 pruebas funcionales con mas de dos keywords de consulta:

In [ ]:
test_queries = [
    "transformación AND sostenible", # Consulta con AND
    "México OR Perú",  # Consulta con OR
    "México AND-NOT Perú",  # Consulta con AND-NOT
    "nonexistent term",  # no debería devolver resultados
]

for query in test_queries:
    print(f"Probando consulta: '{query}'")
    results = apply_boolean_query(query)

    if results.empty:
        print("No se encontraron documentos.")
    else:
        print("Resultados encontrados:")
        print(results[['id', 'text_column']].head())
    print("-" * 50)

## 5. (3 puntos) Actividad Final
- Medir el tiempo de ejecución de las consultas con diferentes tamaños de datos y optimizar el código según sea necesario.


**Entregable:** informe de los resultados obtenidos en formato PDF.